# Crystallography
Anri's crystallography is a set of plain functions on arrays.
A structure with atoms is a `Dans_Diffraction.Crystal`, which also reads the CIF; everything else needs only the lattice parameters and the space-group number.

In [ ]:
import anri.utils
anri.utils.setup()  # before any JAX computation: don't grab most of the GPU memory at start-up

import Dans_Diffraction
import numpy as np

import anri.crystal

In [ ]:
xtl = Dans_Diffraction.Crystal("../../../tests/data/cif/Fe.cif")
lpars = anri.crystal.lattice_parameters(xtl)
sg = anri.crystal.space_group(xtl)
lpars, sg

The B matrix (with $g = U B h$, no $2\pi$).
`B_matrix` evaluates the JAX definition, `anri.crystal.lpars_to_B`, in float64, so there is one definition that can also be jitted and differentiated:

In [ ]:
B = anri.crystal.B_matrix(lpars)
B

Now the reflections up to a $d^*$ limit, with the space group's systematic absences removed.
The wavelength gives their $2\theta$, and caps $d^*$ at $2/\lambda$.
They come sorted by $d^*$:

In [ ]:
dsmax = 1.0
wavelength = 0.18
refl = anri.crystal.reflections(lpars, sg, wavelength, dsmax)
refl["hkl"][:8], refl["ds"][:8], refl["tth"][:8]

Grouping them into rings of equal $d^*$ gives each reflection a ring index, and each ring its $d^*$; counting the members gives the multiplicities:

In [ ]:
ring, ring_ds = anri.crystal.rings(refl["ds"], tol=1e-4)
ring_tth = np.degrees(2 * np.arcsin(ring_ds * wavelength / 2))
ring_mult = np.bincount(ring)
ring_ds, ring_tth, ring_mult

In [ ]:
refl["hkl"][ring == 1]  # the {200} ring

Because we have a structure with atoms, we can compute $|F|^2$ for each reflection, with Debye-Waller factors and anomalous dispersion at this wavelength.
This is where reflections that the space group allows but the atom positions extinguish (e.g. {222} in diamond) show up, as $|F|^2 \approx 0$:

In [ ]:
F2 = anri.crystal.structure_factors(xtl, refl["hkl"], wavelength)
F2_ring = np.array([F2[ring == r].mean() for r in range(len(ring_ds))])
F2_ring

The space group's operations, as 4 x 4 matrices on fractional coordinates, and the rotations of its Laue group in the Cartesian crystal frame (for misorientations and orientation grids):

In [ ]:
S = anri.crystal.symmetry_matrices(sg)
ops = anri.crystal.laue_rotations(S, B)
S.shape, ops.shape

We can confirm the rings against ImageD11 too:

In [ ]:
from ImageD11.unitcell import unitcell

uc = unitcell(lpars, sg)
uc.makerings(dsmax, tol=1e-4)
print(np.abs(ring_ds - np.array(uc.ringds)).max())
print(all(sorted(map(tuple, refl["hkl"][ring == i].tolist())) == sorted(uc.ringhkls[ds]) for i, ds in enumerate(uc.ringds)))